# LangChain: Evaluation

## Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())  # read local .env file

## Create our QandA application

In [2]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch

In [ ]:
file = "OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file)
data = loader.load()

In [ ]:
index = VectorstoreIndexCreator(vectorstore_cls=DocArrayInMemorySearch).from_loaders([
    loader
])

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

# 加载 .env 中的 DEEPSEEK_API_KEY
load_dotenv(find_dotenv())

# 1. 新的导入方式 (使用 langchain_classic)
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA

# 2. 配置 DeepSeek 对话模型
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0,
)

d:\anaconda\envs\ai_agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Administrator\AppData\Local\Temp\ipykernel_25184\2731072551.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [3]:
# 3. 使用本地嵌入模型
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. 加载 CSV 文件
file = "OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file)

# 5. 创建向量索引
index = VectorstoreIndexCreator(
    embedding=embeddings, vectorstore_cls=DocArrayInMemorySearch
).from_loaders([loader])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_25184\77660478.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8092.41it/s]


In [4]:
# llm = ChatOpenAI(temperature=0.0)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs={"document_separator": "<<<<>>>>>"},
)

In [8]:
# 加载后查看文档内容
docs = loader.load()
print(docs[0].page_content)  # 看第一行拼接后的文本

: 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. 

Questions? Please contact us for any inquiries.


In [5]:
data = loader.load()

### Coming up with test datapoints

In [6]:
print(data[10])

page_content=': 10
name: Cozy Comfort Pullover Set, Stripe
description: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.

Size & Fit
- Pants are Favorite Fit: Sits lower on the waist.
- Relaxed Fit: Our most generous fit sits farthest from the body.

Fabric & Care
- In the softest blend of 63% polyester, 35% rayon and 2% spandex.

Additional Features
- Relaxed fit top with raglan sleeves and rounded hem.
- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.

Imported.' metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}


In [8]:
data[11]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

### Hard-coded examples

In [ ]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes",
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection",
    },
]

### LLM-Generated examples

In [9]:
from langchain_classic.evaluation.qa import QAGenerateChain

In [10]:
example_gen_chain = QAGenerateChain.from_llm(llm)

In [11]:
new_examples = example_gen_chain.apply_and_parse([{"doc": t} for t in data[:5]])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_25184\3147780281.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse([{"doc": t} for t in data[:5]])
d:\anaconda\envs\ai_agent\lib\site-packages\langchain_openai\chat_models\base.py:508: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [12]:
new_examples[0]

{'qa_pairs': {'query': "According to the product description, what should a customer do if they wear a half size that is not offered in the Women's Campside Oxfords?",
  'answer': 'For half sizes not offered, the customer should order up to the next whole size.'}}

In [13]:
data[0]

Document(page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.", metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0})

### Combine examples

In [13]:
examples += new_examples

In [35]:
print(examples[0])

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?', 'answer': 'Yes'}


In [14]:
qa.run(examples[0]["query"])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_25184\1223946598.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  qa.run(examples[0]["query"])




> Entering new RetrievalQA chain...

> Finished chain.


'Based on the provided context, there is no information about a "Cozy Comfort Pullover Set." The context only describes features of other items, such as side seam pockets and back zip pockets. Therefore, I don\'t know if the Cozy Comfort Pullover Set has side pockets.'

In [15]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Based on the provided context, there is no information about a "Cozy Comfort Pullover Set." The context only describes features of other items, such as side seam pockets and back zip pockets. Therefore, I don\'t know if the Cozy Comfort Pullover Set has side pockets.'}

## Manual Evaluation

In [21]:
import langchain_classic

langchain_classic.debug = True

In [22]:
qa.run(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


'Based on the provided context, there is no information about a "Cozy Comfort Pullover Set." The context only describes features of other items, such as side seam pockets and back zip pockets. Therefore, I don\'t know if the Cozy Comfort Pullover Set has side pockets.'

In [23]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Based on the provided context, there is no information about a "Cozy Comfort Pullover Set." The context only describes features of other items, such as side seam pockets and back zip pockets. Therefore, I don\'t know if the Cozy Comfort Pullover Set has side pockets.'}

In [26]:
# Turn off the debug mode
langchain.debug = False

## LLM assisted evaluation

In [ ]:
predictions = qa.batch(examples)

In [37]:
# 运行这行，看输出
print("qa 要求的输入键：", qa.input_keys)

qa 要求的输入键： ['query']


In [38]:
print("第一个 example：", examples[0])
print("第一个 example 的 keys：", list(examples[0].keys()))

第一个 example： {'query': 'Do the Cozy Comfort Pullover Set        have side pockets?', 'answer': 'Yes'}
第一个 example 的 keys： ['query', 'answer']


In [42]:
# 遍历所有例子，打印每个的键
for i, ex in enumerate(examples):
    keys = list(ex.keys())
    print(f"第 {i} 个例子的键：{keys}")
    if "query" not in ex:
        print(f"⚠️ 第 {i} 个例子 **缺少 'query' 键**！")

第 0 个例子的键：['query', 'answer']
第 1 个例子的键：['query', 'answer']
第 2 个例子的键：['qa_pairs']
⚠️ 第 2 个例子 **缺少 'query' 键**！
第 3 个例子的键：['qa_pairs']
⚠️ 第 3 个例子 **缺少 'query' 键**！
第 4 个例子的键：['qa_pairs']
⚠️ 第 4 个例子 **缺少 'query' 键**！
第 5 个例子的键：['qa_pairs']
⚠️ 第 5 个例子 **缺少 'query' 键**！
第 6 个例子的键：['qa_pairs']
⚠️ 第 6 个例子 **缺少 'query' 键**！


In [43]:
# 完整流程：清洗 + 推理 + 结果对齐
clean_examples = []
for i, ex in enumerate(examples):
    if "query" in ex:
        clean_examples.append(ex)
    elif "qa_pairs" in ex:
        qa_pairs = ex["qa_pairs"]
        if isinstance(qa_pairs, list):
            clean_examples.extend([
                qa for qa in qa_pairs if isinstance(qa, dict) and "query" in qa
            ])
        elif isinstance(qa_pairs, dict) and "query" in qa_pairs:
            clean_examples.append(qa_pairs)

# 批量推理
predictions = [qa.invoke({"query": ex["query"]}) for ex in clean_examples]

# 结果对齐
results = [
    {"query": ex["query"], "ground_truth": ex["answer"], "prediction": pred["result"]}
    for ex, pred in zip(clean_examples, predictions)
]

print(f"✅ 处理完成：{len(results)} 条有效问答对")



> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.
✅ 处理完成：7 条有效问答对


In [30]:
from langchain_classic.evaluation.qa import QAEvalChain

In [31]:
# llm = ChatOpenAI(temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

In [44]:
graded_outputs = eval_chain.evaluate(examples, predictions)

KeyError: 'query'

In [45]:
# 错误写法（你现在的）
# graded_outputs = eval_chain.evaluate(examples, predictions)

# 正确写法（唯一解法）
graded_outputs = eval_chain.evaluate(clean_examples, predictions)

d:\anaconda\envs\ai_agent\lib\site-packages\langchain_openai\chat_models\base.py:508: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [46]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]["query"])
    print("Real Answer: " + predictions[i]["answer"])
    print("Predicted Answer: " + predictions[i]["result"])
    print("Predicted Grade: " + graded_outputs[i]["text"])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?


KeyError: 'answer'

In [ ]:
for i in range(len(graded_outputs)):
    print("=" * 60)
    print(f"问题: {clean_examples[i]['query']}")
    print(f"标准答案: {clean_examples[i]['answer']}")
    print(f"模型回答: {predictions[i]['result']}")
    # 这里改成 results 就对了！
    print(f"评估结果: {graded_outputs[i]['results']}")

问题: Do the Cozy Comfort Pullover Set        have side pockets?
标准答案: Yes
模型回答: Based on the provided context, there is no information about a "Cozy Comfort Pullover Set." The context only describes features of other items, such as side seam pockets and back zip pockets on a different product. Therefore, I don't know if the Cozy Comfort Pullover Set has side pockets.


KeyError: 'text'